# Project 2: Customer Churn Prediction — Machine Learning & Evaluation

## 📌 Objective
Predict whether customers will leave a telecom service provider using exploratory analysis, feature engineering, Logistic Regression, and Random Forest classification models.

--- 

### 📁 Project Structure
- `raw_data/`: Raw Telco customer records
- `cleaned_data/`: Cleaned dataset free of missing values and duplicates
- `output/`: Confusion Matrix, ROC-AUC curve, and Feature Importance charts
- `churn_model.pkl`: Serialized trained model & scalar pipeline

## Step 1: Data Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw data
df_raw = pd.read_csv('raw_data/telco_customer_churn_raw.csv')
print('Raw Dataset Shape:', df_raw.shape)
df_raw.head()

## Step 2: Data Cleaning & Standardizing

In [ ]:
# 1. Drop duplicates
df = df_raw.drop_duplicates().copy()

# 2. Drop unwanted customerID
if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)

# 3. Handle missing values in TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 4. Standardize Contract & InternetService categories
df['Contract'] = df['Contract'].str.lower().replace({'month-to-month': 'month-to-month', 'one year': 'one-year', 'two year': 'two-year'})
df['InternetService'] = df['InternetService'].str.lower()
df['Churn_Binary'] = df['Churn'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

print('Cleaned Shape:', df.shape)
df.head()

## Step 3: Feature Engineering & Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

df['avg_monthly_per_tenure'] = df['TotalCharges'] / (df['tenure'] + 1)
df['is_high_charge'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_per_tenure', 'is_high_charge']

df_encoded = df.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

X = df_encoded[num_cols + cat_cols]
y = df_encoded['Churn_Binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 4: Model Building & Priority Metrics Evaluation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
FPR = FP / (FP + TN)
TPR = TP / (TP + FN)
FNR = FN / (TP + FN)
auc = roc_auc_score(y_test, y_prob)

print('=== EVALUATION METRICS ===')
print(f'Recall (TPR):        {TPR:.4f}')
print(f'False Positive Rate: {FPR:.4f}')
print(f'False Negative Rate: {FNR:.4f} (Key Retention Metric)')
print(f'ROC-AUC Score:       {auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stay', 'Churn']))

## Step 5: Visualizations (Confusion Matrix & ROC-AUC)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# ROC Curve
fpr_vals, tpr_vals, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr_vals, tpr_vals, color='orange', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_title('ROC-AUC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.show()